# 특약 모형 ④ 일별 현금흐름

치료 경로에 사망·지급 조건·월납·반환·사업비를 연결한다. 월 보험료는 아직 미정이므로 1원당 계수와 정액 지출을 구분한다.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
root = Path.cwd()
if not (root / "models").exists():
    root = root.parent
if not (root / "models").exists():
    raise RuntimeError("저장소 루트 또는 notebooks 폴더에서 실행하세요.")
sys.path.insert(0, str(root))
from models.rider.cashflows import CashflowSettings, project_cashflows, summarize
from models.rider.start_integration import read_starts, read_mortality
from models.treatment.model import Parameters
settings = CashflowSettings(**json.loads((root / "config/cashflows.json").read_text()))
treatment = Parameters(**json.loads((root / "config/treatment.json").read_text()))
settings

In [ ]:
starts = read_starts(root / "outputs/start_rate/start_probabilities.csv", "base")
rows = project_cashflows(starts, read_mortality(), settings, treatment)
daily = pd.DataFrame(rows)
summary = summarize(rows, settings.initial_policies)
daily.head()

## 계약 흐름

소진 이후의 사망은 특약 계약 흐름에서 제외된다. 최종 소진·사망·만기 인원의 합이 최초 가입자 수와 일치하는지 확인한다.

In [ ]:
pd.Series({k: summary[k] for k in ["total_starts", "total_paid_claims", "exhausted", "deaths", "matured", "max_mass_error"]})

## 지출과 보험료 계수

정액 지출의 현재가치와 G 1원당 순유입 현재가치를 준비한다. 보험료 결정은 다음 단계다.

In [ ]:
pd.Series({k: v for k, v in summary.items() if k.startswith("pv_")})

In [ ]:
from datetime import date
contract_start = date.fromisoformat(starts[0]["date"])
calendar_dates = daily["date"].map(date.fromisoformat)
daily["policy_year"] = calendar_dates.map(lambda d: d.year-contract_start.year-int((d.month,d.day)<(contract_start.month,contract_start.day))+1)
annual = daily.groupby("policy_year")[["new_starters","paid_claims","exhausted_today","deaths_today","premium_units","refund_units","benefit_krw","fixed_outflow_krw"]].sum()
annual
